Plots

Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import plotly.io as pio


# Input and output directories
processed_dir = r"S:\path\StructureMap\plots"  #place the plots folder path
plots_dir = r"S:\path\StructureMap\plots"   #place the plots folder path

os.makedirs(plots_dir, exist_ok=True)

print("Plots will be saved in:", plots_dir)


In [ ]:
# Subfolders for each plot type
subfolders = {
    "pPSE_PTMs_1D": "pPSE_PTMs_1D",
    "PTM_distribution": "PTM_distribution",
    "pPSE_3D": "pPSE_3D",
    "pLDDT_3D": "pLDDT_3D",
    "PTMs_3D": "PTMs_3D",
    "domains_1D": "domains_1D",
    "domains_3D": "domains_3D",
    "IDR_3D": "IDR_3D",
    "IDR_1D": "IDR_1D",
    "pPSE_PTMs_3D": "pPSE_PTMs_3D"
}

# Create all subfolders
for key, folder in subfolders.items():
    path = os.path.join(plots_dir, folder)
    os.makedirs(path, exist_ok=True)
    subfolders[key] = path

subfolders


Load input table

In [ ]:
alphafold_ptms = pd.read_csv(
    os.path.join(processed_dir, "alphafold_ptms.tsv"),
    sep="\t"
)

alphafold_ptms.head()


Define protein list

In [ ]:
protein_list = ['O55143','O70468','P05202','P16858','P48962','Q02566']

Generate PTM table (long format)

In [ ]:
def generate_ptm_table(df_prot, ptms):
    """
    Convert wide PTM columns into a long-format table grouped by residue.
    """

    df_long = df_prot.melt(
        id_vars=["protein_id", "position", "AA", "nAA_12_70_pae"],
        value_vars=ptms,
        var_name="PTM",
        value_name="value"
    )

    # Keep only present PTMs
    df_long = df_long[df_long["value"] == 1]

    if df_long.empty:
        return df_long

    # Create residue label (e.g., S145)
    df_long["Residue"] = df_long["AA"] + df_long["position"].astype(str)

    # Group PTMs per residue
    df_grouped = (
        df_long
        .groupby(["protein_id", "position", "AA", "nAA_12_70_pae", "Residue"])["PTM"]
        .apply(lambda x: ", ".join(sorted(x)))
        .reset_index()
    )

    # Count number of PTMs
    df_grouped["n_ptms"] = df_grouped["PTM"].str.count(",") + 1

    return df_grouped


ptms = ["p","me","ox","diox","triox","kin","rev_ox","carb"]
ptm_table = generate_ptm_table(alphafold_ptms, ptms)
ptm_table.head()


pPSE + PTM plot 1D (per protein)

In [ ]:
import plotly.graph_objects as go

def plot_pPSE_PTMs(df_all, ptm_table, protein_list):

    ptm_colors = {
        "p": "#F4B400", "me": "#1E88E5", "ox": "#E53935",
        "diox": "#8E24AA", "triox": "#EC407A", "rev_ox": "#546E7A",
        "carb": "#6D4C41", "kin": "#00897B"
    }
    multi_ptm_color = "#9B80DA"

    for prot in protein_list:

        df_prot_all = df_all[df_all["protein_id"] == prot].sort_values("position")
        df_prot_ptm = ptm_table[ptm_table["protein_id"] == prot]

        fig = go.Figure()

        fig.add_trace(go.Scatter(
            x=df_prot_all["position"],
            y=df_prot_all["nAA_12_70_pae"],
            mode="lines",
            line=dict(color="#C1E078", width=2),
            name="pPSE"
        ))

        fig.add_hline(
            y=5,
            line=dict(color="#412073", width=1, dash="dash"),
            annotation_text="pPSE = 5"
        )

        df_single = df_prot_ptm[df_prot_ptm["n_ptms"] == 1]
        df_multi = df_prot_ptm[df_prot_ptm["n_ptms"] > 1]

        for ptm_type in df_single["PTM"].unique():
            df_ptm = df_single[df_single["PTM"] == ptm_type]
            fig.add_trace(go.Scatter(
                x=df_ptm["position"],
                y=df_ptm["nAA_12_70_pae"],
                mode="markers",
                marker=dict(size=10, color=ptm_colors[ptm_type]),
                name=ptm_type
            ))

        if not df_multi.empty:
            fig.add_trace(go.Scatter(
                x=df_multi["position"],
                y=df_multi["nAA_12_70_pae"],
                mode="markers",
                marker=dict(size=14, color=multi_ptm_color),
                name="multiple PTMs"
            ))

        fig.update_layout(
            title=f"{prot} — pPSE and PTM sites",
            width=1900, height=500, template="simple_white"
        )

        out = os.path.join(subfolders["pPSE_PTMs_1D"], f"{prot}_pPSE_PTMs_1D.html")
        fig.write_html(out)
        print("Saved:", out)


plot_pPSE_PTMs(alphafold_ptms, ptm_table, protein_list)


3D pPSE visualization

In [ ]:
import py3Dmol

def plot_3D_pPSE(prot, df_prot):

    cif_path = os.path.join(cif_folder, f"{prot}.cif")
    if not os.path.exists(cif_path):
        print("Missing CIF:", prot)
        return

    with open(cif_path) as f:
        cif_data = f.read()

    view = py3Dmol.view(width=1600, height=1200)
    view.addModel(cif_data, "cif")
    view.setStyle({"cartoon": {"color": "white"}})

    for _, row in df_prot.iterrows():
        resi = str(int(row["position"]))
        pPSE = row["nAA_12_70_pae"]

        if pPSE == 0: color = "0xAC63D7"
        elif pPSE <= 5: color = "0xC1CA60"
        elif pPSE <= 8: color = "0xCA6760"
        else: color = "0x6688CC"

        view.setStyle({"resi": resi}, {"cartoon": {"color": color}})

    view.zoomTo()

    html = view._make_html()
    out = os.path.join(subfolders["pPSE_3D"], f"{prot}_3D_pPSE.html")
    with open(out, "w") as f:
        f.write(html)

    print("Saved:", out)


cif_folder = r"C:\Users\user\AppData\Local\Temp\cif" #replace with username

for prot in protein_list:
    df_prot = alphafold_ptms[alphafold_ptms["protein_id"] == prot]
    plot_3D_pPSE(prot, df_prot)


3D pLDDT visualization

In [ ]:
import gemmi

def extract_plddt(cif_path):
    doc = gemmi.cif.read_file(cif_path)
    block = doc.sole_block()
    structure = gemmi.make_structure_from_block(block)

    positions, plddt_values = [], []

    for model in structure:
        for chain in model:
            for residue in chain:
                if len(residue) > 0:
                    positions.append(residue.seqid.num)
                    plddt_values.append(residue[0].b_iso)

    return pd.DataFrame({"position": positions, "pLDDT": plddt_values})


In [ ]:
def plot_3D_pLDDT(prot):

    cif_path = os.path.join(cif_folder, f"{prot}.cif")
    df = extract_plddt(cif_path)

    with open(cif_path) as f:
        cif_data = f.read()

    view = py3Dmol.view(width=1600, height=1200)
    view.addModel(cif_data, "cif")
    view.setStyle({"cartoon": {"color": "white"}})

    for _, row in df.iterrows():
        resi = str(int(row["position"]))
        plddt = row["pLDDT"]

        if plddt <= 50: color = "0xF08E1F"
        elif plddt <= 70: color = "0xE8D641"
        elif plddt <= 90: color = "0x7EE3E6"
        else: color = "0x6191E8"

        view.setStyle({"resi": resi}, {"cartoon": {"color": color}})

    view.zoomTo()

    html = view._make_html()
    out = os.path.join(subfolders["pLDDT_3D"], f"{prot}_3D_pLDDT.html")
    with open(out, "w") as f:
        f.write(html)

    print("Saved:", out)


for prot in protein_list:
    plot_3D_pLDDT(prot)


3D PTM visualization

In [ ]:
ptm_color_map = {
    "p": "0xF4B400", "me": "0x1E88E5", "ox": "0xE53935",
    "diox": "0x8E24AA", "triox": "0xEC407A", "rev_ox": "0x546E7A",
    "carb": "0x6D4C41", "kin": "0x00897B"
}


In [ ]:
def plot_3D_PTMs(prot, df_prot):

    cif_path = os.path.join(cif_folder, f"{prot}.cif")
    if not os.path.exists(cif_path):
        print("Missing CIF:", prot)
        return

    with open(cif_path) as f:
        cif_data = f.read()

    view = py3Dmol.view(width=1600, height=1200)
    view.addModel(cif_data, "cif")
    view.setStyle({"cartoon": {"color": "white"}})

    for _, row in df_prot.iterrows():
        resi = int(row["position"])
        for ptm in ptm_color_map:
            if row.get(ptm, 0) == 1:
                view.addStyle(
                    {"resi": str(resi), "atom": "CA"},
                    {"sphere": {"color": ptm_color_map[ptm], "radius": 1.4}}
                )

    view.zoomTo()

    html = view._make_html()
    out = os.path.join(subfolders["PTMs_3D"], f"{prot}_3D_PTMs.html")
    with open(out, "w") as f:
        f.write(html)

    print("Saved:", out)


for prot in protein_list:
    df_prot = alphafold_ptms[alphafold_ptms["protein_id"] == prot]
    plot_3D_PTMs(prot, df_prot)


1D domain + PTM map

In [ ]:
import json

def extract_domains(json_path):
    with open(json_path) as f:
        data = json.load(f)

    features = []
    for feat in data["features"]:
        if feat["type"] in ["Domain","Region","Coiled coil","Binding site","Repeat","Motif","Site","Modified residue","Disulfide bond"]:
            features.append({
                "type": feat["type"],
                "name": feat.get("description", feat["type"]),
                "start": int(feat["location"]["start"]["value"]),
                "end": int(feat["location"]["end"]["value"])
            })
    return features


In [ ]:
features_dict = {}
for prot in protein_list:
    json_path = fr"C:\path\data\processed\domains\domains_{prot}.json"
    features_dict[prot] = extract_domains(json_path)


In [ ]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

def plot_protein_map_interactive(df_prot, features, protein_name, output_path=None):
    """
    df_prot: dataframe with columns 'position', 'p','me','ox'
    features: list of dictionaries with keys 'type','name','start','end'
    protein_name: string
    """
    print("Output path received:", output_path)

    # --- Initialize figure ---
    fig = go.Figure()

    protein_length = df_prot["position"].max()

    # --- Base protein line ---
    fig.add_trace(go.Scatter(
        x=[1, protein_length],
        y=[0, 0],
        mode="lines",
        line=dict(color="lightgrey", width=10),
        showlegend=False
    ))

    # --- Draw domains and regions ---
    feature_colors = {"Domain": "#82B5AF", "Region": "#B58288", "Coiled coil": "#8288B5","Binding site": "#8BA74A", "Repeat": "#5C01B1", "Motif": "#0156B1" , "Site": "#B1401D", "Modified residue": "#F5D64E", "Disulfide bond": "#F0911A" }
    
    last_x = -999
    y_base = 0.25
    offset_step = 0.18
    min_distance = 35 


    for f in features:
        fig.add_shape(
            type="rect",
            x0=f["start"],
            x1=f["end"],
            y0=-0.2,
            y1=0.2,
            line=dict(color="black", width=1),
            fillcolor=feature_colors.get(f["type"], "#CCCCCC"),
            opacity=0.6
        )
        if abs(f["start"] - last_x) < min_distance:
            y_base += offset_step
        else:
            y_base = 0.25
            
        fig.add_annotation(
            x=(f["start"] + f["end"]) / 2,
            y=0.25,
            text=f["name"],
            showarrow=False,
            font=dict(size=8),
            textangle=80
        )

        last_x = f["start"]

    # --- PTMs ---
    ptm_colors = {
    "p": "#F4B400",      # ambar
    "me": "#1E88E5",     # blue
    "ox": "#E53935",     # red
    "diox": "#8E24AA",   # lilac
    "triox": "#EC407A",  # pink
    "rev_ox": "#546E7A", # grey
    "carb": "#6D4C41",   # brown
    "kin": "#00897B"     # dark green
    }

    ptm_cols = [ptm for ptm in ptm_colors.keys() if ptm in df_prot.columns]


    # Group by residue to handle multiple PTMs
    df_grouped = df_prot.groupby("position", as_index=False)[ptm_cols].max()

    ptm_positions = {ptm: [] for ptm in ptm_cols}
    ptm_offsets = {ptm: [] for ptm in ptm_cols}


    for _, row in df_grouped.iterrows():
        pos = row["position"]

        active_ptms = [ptm for ptm in ptm_cols if row[ptm] == 1]

        if not active_ptms:
            continue

        offsets = np.linspace(-0.2, 0.2, len(active_ptms))

        for ptm, offset in zip(active_ptms, offsets):
            ptm_positions[ptm].append(pos)
            ptm_offsets[ptm].append(1 + offset)
        
    for ptm in ptm_cols:
        if len(ptm_positions[ptm]) == 0:
            continue

        fig.add_trace(go.Scatter(
            x=ptm_positions[ptm],
            y=ptm_offsets[ptm],
            mode="markers",
            marker=dict(color=ptm_colors[ptm], size=10),
            name=ptm.upper(),
            hovertemplate=(
                "Residue: %{x}<br>"
                f"PTM: {ptm.upper()}<extra></extra>"
            )
            
        ))
    

    # --- Layout ---
    fig.update_layout(
        title=f"Protein {protein_name} - Domains, Regions & PTMs",
        xaxis_title="Residue position",
        yaxis=dict(visible=False),
        width=1900,
        height=500,
        template="simple_white"
    )


    out = os.path.join(subfolders["domains_1D"], f"{prot}_domains_PTMs_1D.html")
    fig.write_html(out)
    print("Saved:", out)


In [ ]:
for prot in protein_list:
    df_prot = alphafold_ptms[alphafold_ptms["protein_id"] == prot]
    plot_protein_map_interactive(df_prot, features_dict[prot], prot)


3D domain +  PTM map

In [ ]:
import os
import py3Dmol

def plot_protein_3D_domains_PTMs(prot, df_prot, features, cif_folder, output_folder):
    """
    3D visualization of:
    - Domains (colored regions)
    - PTMs (colored spheres)
    - Multi-PTM residues (purple spheres)
    """

    # ---- Load CIF ----
    cif_path = os.path.join(cif_folder, f"{prot}.cif")
    if not os.path.exists(cif_path):
        print(f"CIF not found for {prot}")
        return

    with open(cif_path) as f:
        cif_data = f.read()

    # ---- Create viewer ----
    view = py3Dmol.view(width=1600, height=1200)
    view.addModel(cif_data, "cif")
    view.setStyle({"cartoon": {"color": "white"}})

    # ---- Hover labels ----
    view.setHoverable(
        {"chain": "A"},
        True,
        '''
        function(atom,viewer,event,container) {
            if(!atom.label) {
                atom.label = viewer.addLabel(
                    atom.resn + " " + atom.resi,
                    {
                        position: atom,
                        backgroundColor: 'mintcream',
                        fontColor: 'black',
                        fontSize: 14,
                        padding: 4
                    }
                );
            }
        }
        ''',
        '''
        function(atom,viewer) {
            if(atom.label) {
                viewer.removeLabel(atom.label);
                delete atom.label;
            }
        }
        '''
    )

    # ---- Domain colors (py3Dmol format: 0xRRGGBB) ----
    domain_colors = {
        "Domain": "0x82B5AF",
        "Region": "0xB58288",
        "Coiled coil": "0x8288B5",
        "Binding site": "0x8BA74A",
        "Repeat": "0x5C01B1",
        "Motif": "0x0156B1",
        "Site": "0xB1401D",
        "Modified residue": "0xF5D64E",
        "Disulfide bond": "0xF0911A"
    }

    # ---- Color domains ----
    for feat in features:
        if feat["type"] in domain_colors:
            view.setStyle(
                {"chain": "A", "resi": list(range(feat["start"], feat["end"] + 1))},
                {"cartoon": {"color": domain_colors[feat["type"]]}}
            )

    # ---- PTM colors ----
    ptm_colors = {
        "p": "0xF4B400",
        "me": "0x1E88E5",
        "ox": "0xE53935",
        "diox": "0x8E24AA",
        "triox": "0xEC407A",
        "rev_ox": "0x546E7A",
        "carb": "0x6D4C41",
        "kin": "0x00897B"
    }

    # ---- Multi-PTM detection ----
    ptm_cols = [ptm for ptm in ptm_colors if ptm in df_prot.columns]
    df_grouped = df_prot.groupby("position", as_index=False)[ptm_cols].max()
    df_grouped["ptm_count"] = df_grouped[ptm_cols].sum(axis=1)
    multi_ptm_residues = df_grouped.loc[df_grouped["ptm_count"] > 1, "position"].tolist()

    # ---- Add PTM spheres ----
    for _, row in df_prot.iterrows():
        resi = int(row["position"])

        # Multi-PTM → purple sphere
        if resi in multi_ptm_residues:
            view.addStyle(
                {"chain": "A", "resi": str(resi), "atom": "CA"},
                {"sphere": {"color": "0x4F45A6", "radius": 2.5}}
            )
            continue

        # Single PTMs
        for ptm, color in ptm_colors.items():
            if row.get(ptm, 0) == 1:
                view.addStyle(
                    {"chain": "A", "resi": str(resi), "atom": "CA"},
                    {"sphere": {"color": color, "radius": 1.4}}
                )

    view.zoomTo()

    # ---- Save HTML ----
    os.makedirs(output_folder, exist_ok=True)
    out = os.path.join(output_folder, f"{prot}_domains_PTMs_3D.html")

    with open(out, "w") as f:
        f.write(view._make_html())

    print(f"Saved 3D domain+PTM visualization for {prot} → {out}")


In [ ]:
for prot in protein_list:
    df_prot = alphafold_ptms[alphafold_ptms["protein_id"] == prot]
    features = features_dict[prot]
    plot_protein_3D_domains_PTMs(
        prot,
        df_prot,
        features,
        cif_folder,
        subfolders["domains_3D"]
    )


IDRs 1D

In [ ]:
import plotly.graph_objects as go

def plot_IDR_1D(df_prot, prot, output_folder):
    """
    1D plot showing:
    - pPSE line
    - Long IDRs
    - Short IDRs
    - Structured regions
    """

    fig = go.Figure()

    # pPSE line
    fig.add_trace(go.Scatter(
        x=df_prot["position"],
        y=df_prot["nAA_12_70_pae"],
        mode="lines",
        line=dict(color="#6AA84F", width=2),
        name="pPSE"
    ))

    # Long IDRs
    long_idr = df_prot[df_prot["IDR"] == 1]
    fig.add_trace(go.Scatter(
        x=long_idr["position"],
        y=[0]*len(long_idr),
        mode="markers",
        marker=dict(color="#F4B400", size=6),
        name="Long IDR"
    ))

    # Short IDRs
    short_idr = df_prot[df_prot["flexible_pattern"] == 1]
    fig.add_trace(go.Scatter(
        x=short_idr["position"],
        y=[0]*len(short_idr),
        mode="markers",
        marker=dict(color="#E53935", size=6),
        name="Short IDR"
    ))

    # Structured regions
    structured = df_prot[df_prot["IDR"] == 0]
    fig.add_trace(go.Scatter(
        x=structured["position"],
        y=[0]*len(structured),
        mode="markers",
        marker=dict(color="#1E88E5", size=4),
        name="Structured"
    ))

    fig.update_layout(
        title=f"{prot} — IDR map (1D)",
        xaxis_title="Residue position",
        yaxis_title="pPSE",
        width=1900,
        height=500,
        template="simple_white"
    )

    out = os.path.join(output_folder, f"{prot}_IDR_1D.html")
    fig.write_html(out)
    print("Saved:", out)


In [ ]:
for prot in protein_list:
    df_prot = alphafold_ptms[alphafold_ptms["protein_id"] == prot]
    plot_IDR_1D(df_prot, prot, subfolders["IDR_1D"])


IDRs 3D

In [ ]:
import py3Dmol

def plot_IDR_3D(prot, df_prot, cif_folder, output_folder):
    """
    3D visualization of:
    - Long IDRs
    - Short IDRs
    - Structured regions
    """

    cif_path = os.path.join(cif_folder, f"{prot}.cif")
    if not os.path.exists(cif_path):
        print("CIF not found:", prot)
        return

    with open(cif_path) as f:
        cif_data = f.read()

    view = py3Dmol.view(width=1600, height=1200)
    view.addModel(cif_data, "cif")

    # Base style
    view.setStyle({"cartoon": {"color": "white", "opacity": 0.3}})

    # Long IDRs
    long_idr = df_prot[df_prot["IDR"] == 1]["position"].tolist()
    if long_idr:
        view.setStyle(
            {"resi": long_idr, "chain": "A"},
            {"cartoon": {"color": "0xF4B400"}}
        )

    # Short IDRs
    short_idr = df_prot[df_prot["flexible_pattern"] == 1]["position"].tolist()
    if short_idr:
        view.setStyle(
            {"resi": short_idr, "chain": "A"},
            {"cartoon": {"color": "0xE53935"}}
        )

    # Structured regions
    structured = df_prot[df_prot["IDR"] == 0]["position"].tolist()
    if structured:
        view.setStyle(
            {"resi": structured, "chain": "A"},
            {"cartoon": {"color": "0x1E88E5"}}
        )

    view.zoomTo()

    html = view._make_html()
    out = os.path.join(output_folder, f"{prot}_IDR_3D.html")
    with open(out, "w") as f:
        f.write(html)

    print("Saved:", out)


In [ ]:
for prot in protein_list:
    df_prot = alphafold_ptms[alphafold_ptms["protein_id"] == prot]
    plot_IDR_3D(prot, df_prot, cif_folder, subfolders["IDR_3D"])


pPSE + PTMs 3D

In [ ]:
import py3Dmol
import os

def plot_3D_pPSE_PTMs(prot, df_prot, cif_folder, output_folder):
    """
    3D visualization combining:
    - pPSE (cartoon color)
    - PTMs (colored spheres)
    - Multi-PTM residues (purple spheres)
    """

    # ---- Load CIF ----
    cif_path = os.path.join(cif_folder, f"{prot}.cif")
    if not os.path.exists(cif_path):
        print("CIF not found:", prot)
        return

    with open(cif_path) as f:
        cif_data = f.read()

    # ---- Create viewer ----
    view = py3Dmol.view(width=1600, height=1200)
    view.addModel(cif_data, "cif")

    # ---- Base style ----
    view.setStyle({"cartoon": {"color": "white"}})

    # ---- Color cartoon by pPSE ----
    for _, row in df_prot.iterrows():
        resi = str(int(row["position"]))
        pPSE = row["nAA_12_70_pae"]

        if pPSE == 0:
            color = "0xAC63D7"
        elif pPSE <= 5:
            color = "0xC1CA60"
        elif pPSE <= 8:
            color = "0xCA6760"
        else:
            color = "0x6688CC"

        view.setStyle({"resi": resi}, {"cartoon": {"color": color}})

    # ---- PTM colors ----
    ptm_colors = {
        "p": "0xF4B400",
        "me": "0x1E88E5",
        "ox": "0xE53935",
        "diox": "0x8E24AA",
        "triox": "0xEC407A",
        "rev_ox": "0x546E7A",
        "carb": "0x6D4C41",
        "kin": "0x00897B"
    }

    ptm_cols = [ptm for ptm in ptm_colors if ptm in df_prot.columns]

    # ---- Detect multi-PTM residues ----
    df_grouped = df_prot.groupby("position", as_index=False)[ptm_cols].max()
    df_grouped["ptm_count"] = df_grouped[ptm_cols].sum(axis=1)
    multi_ptm_residues = df_grouped.loc[df_grouped["ptm_count"] > 1, "position"].tolist()

    # ---- Add PTM spheres ----
    for _, row in df_prot.iterrows():
        resi = int(row["position"])

        # Multi-PTM → purple sphere
        if resi in multi_ptm_residues:
            view.addStyle(
                {"resi": str(resi), "atom": "CA"},
                {"sphere": {"color": "0x4F45A6", "radius": 2.5}}
            )
            continue

        # Single PTMs
        for ptm, color in ptm_colors.items():
            if row.get(ptm, 0) == 1:
                view.addStyle(
                    {"resi": str(resi), "atom": "CA"},
                    {"sphere": {"color": color, "radius": 1.4}}
                )

    # ---- Hover labels ----
    view.setHoverable(
        {"chain": "A"},
        True,
        '''
        function(atom,viewer,event,container) {
            if(!atom.label) {
                atom.label = viewer.addLabel(
                    atom.resn + " " + atom.resi,
                    {
                        position: atom,
                        backgroundColor: 'mintcream',
                        fontColor: 'black',
                        fontSize: 14,
                        padding: 4
                    }
                );
            }
        }
        ''',
        '''
        function(atom,viewer) {
            if(atom.label) {
                viewer.removeLabel(atom.label);
                delete atom.label;
            }
        }
        '''
    )

    view.zoomTo()

    # ---- Save HTML ----
    os.makedirs(output_folder, exist_ok=True)
    out = os.path.join(output_folder, f"{prot}_pPSE_PTMs_3D.html")

    with open(out, "w") as f:
        f.write(view._make_html())

    print(f"Saved 3D pPSE+PTMs visualization for {prot} → {out}")


In [ ]:
for prot in protein_list:
    df_prot = alphafold_ptms[alphafold_ptms["protein_id"] == prot]
    plot_3D_pPSE_PTMs(
        prot,
        df_prot,
        cif_folder,
        subfolders["pPSE_PTMs_3D"]
    )
